# 🔒 Measure Privacy: Simple Attacks and Risk Metrics

In this lesson we will learn how to estimate how much information from the original dataset may have leaked into the synthetic one, using metrics like exact duplicate checks and nearest-neighbour distance analysis.

Privacy evaluation for synthetic data is not about identifiers like names or IDs: those should be removed and/or masked before training ever starts. It is about what we call the **quasi-identifiers** - attributes like age, location, or job that, when combined, can point back to a real individual.

**In this notebook, we will:**
1. Check for exact duplicates between original and synthetic data
2. Measure similarity using the Nearest Neighbour Distance Ratio (NNDR)

## 1. Load the Datasets

In [4]:
import pandas as pd

original_df  = pd.read_csv('../data/bank-full.csv', sep=';')
synthetic_df = pd.read_csv('../data/bank-full_synthetic.csv')

print(f'Original : {original_df.shape[0]:,} rows × {original_df.shape[1]} columns')
print(f'Synthetic: {synthetic_df.shape[0]:,} rows × {synthetic_df.shape[1]} columns')

Original : 45,211 rows × 17 columns
Synthetic: 45,211 rows × 17 columns


In [5]:
original_df = original_df.drop('duration', axis=1) # removing the calculated feature created
synthetic_df = synthetic_df.drop('total_contacts', axis=1) # removing the calculated feature created

## 2. Exact Duplicate Check

The simplest and most obvious privacy check: are any synthetic records an exact copy of a real one? A well-trained synthesizer should not reproduce real records verbatim, unless those are extremely common records.

Remeber that the original dataset itself had 39 duplicate rows.

In [7]:
# Find rows in synthetic that appear exactly in original
shared_cols = [c for c in original_df.columns if c in synthetic_df.columns]

original_tuples  = set(original_df[shared_cols].apply(tuple, axis=1))
synthetic_tuples = synthetic_df[shared_cols].apply(tuple, axis=1)

exact_matches = synthetic_tuples.isin(original_tuples)

n_duplicates = exact_matches.sum()
pct          = n_duplicates / len(synthetic_df) * 100

print(f'Exact duplicates : {n_duplicates:,} / {len(synthetic_df):,} synthetic records ({pct:.2f}%)')
print('→ ' + ('No exact duplicates — good.' if n_duplicates == 0
               else f'{pct:.2f}% of synthetic records are verbatim copies of real records.'))

Exact duplicates : 2,611 / 45,211 synthetic records (5.78%)
→ 5.78% of synthetic records are verbatim copies of real records.


Exact duplicates are only a concern when the matching real record is unique in the original dataset. If multiple real individuals share the same combination of attributes, a synthetic record matching them cannot be traced back to any single person, that is the hiding in the crowd effect, and it is not a privacy risk.

To understand whether a duplicate is risky, check how many times that combination of values appears in the original data. A match to a record that appears once is a red flag. A match to a record that appears ten times is not.

## 3. Nearest Neighbour Distance Ratio (NNDR)

For each synthetic record, we find its two closest real records and compute the ratio of their distances:

$$\text{NNDR} = \frac{\text{distance to closest real record}}{\text{distance to second closest real record}}$$

- **NNDR close to 0** — the synthetic record is much closer to one real record than any other. This suggests the synthesiser may have memorised that individual.
- **NNDR close to 1** — the synthetic record is equidistant from multiple real records. It is hiding in the crowd — no single individual can be singled out.

> **Note:** We run this on a sample of 500 records from each dataset. Computing nearest neighbours over the full 45,000-row dataset is computationally intensive and unnecessary for illustration — the pattern in a representative sample is the same.

In [8]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

SAMPLE_SIZE = 500

def preprocess(df, shared_cols):
    """Encode categoricals and normalise numerics to [0, 1]."""
    df = df[shared_cols].copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    df = pd.DataFrame(MinMaxScaler().fit_transform(df), columns=df.columns)
    return df

orig_sample  = original_df[shared_cols].sample(SAMPLE_SIZE, random_state=42)
synth_sample = synthetic_df[shared_cols].sample(SAMPLE_SIZE, random_state=42)

orig_enc  = preprocess(orig_sample, shared_cols)
synth_enc = preprocess(synth_sample, shared_cols)

print(f'Sample prepared: {SAMPLE_SIZE} original + {SAMPLE_SIZE} synthetic records ✓')

Sample prepared: 500 original + 500 synthetic records ✓


In [9]:
# Fit on real data, query with synthetic
nn = NearestNeighbors(n_neighbors=2, metric='euclidean')
nn.fit(orig_enc)

distances, _ = nn.kneighbors(synth_enc)

# NNDR = distance to 1st neighbour / distance to 2nd neighbour
# Add a small epsilon to avoid division by zero
nndr = distances[:, 0] / (distances[:, 1] + 1e-10)

nndr_series = pd.Series(nndr, name='nndr')
print('NNDR distribution:')
print(nndr_series.describe().round(4))

NNDR distribution:
count    500.0000
mean       0.8018
std        0.1946
min        0.0679
25%        0.7065
50%        0.8675
75%        0.9496
max        0.9995
Name: nndr, dtype: float64


In [10]:
# Risk summary
HIGH_RISK_THRESHOLD = 0.1   # very close to one real record

high_risk = (nndr_series < HIGH_RISK_THRESHOLD).sum()
pct_risk  = high_risk / len(nndr_series) * 100

print(f'Records with NNDR < {HIGH_RISK_THRESHOLD} : {high_risk} ({pct_risk:.1f}%)')
print(f'Mean NNDR                           : {nndr_series.mean():.4f}')
print(f'Min  NNDR                           : {nndr_series.min():.4f}')
print()
print('→ ' + ('Low risk — most synthetic records are not close to any single real record.'
               if pct_risk < 5 else
               f'{pct_risk:.1f}% of synthetic records are very close to a real one — review these.'))

Records with NNDR < 0.1 : 1 (0.2%)
Mean NNDR                           : 0.8018
Min  NNDR                           : 0.0679

→ Low risk — most synthetic records are not close to any single real record.


In [11]:
# Inspect the closest matches
closest_idx = nndr_series.nsmallest(5).index
print('5 synthetic records closest to a real record (lowest NNDR):')
print(pd.DataFrame({
    'nndr':              nndr_series[closest_idx].round(4).values,
    'dist_to_closest':   distances[closest_idx, 0].round(4),
    'dist_to_2nd':       distances[closest_idx, 1].round(4),
}))

5 synthetic records closest to a real record (lowest NNDR):
     nndr  dist_to_closest  dist_to_2nd
0  0.0679           0.0490       0.7218
1  0.1606           0.0716       0.4459
2  0.1634           0.0786       0.4814
3  0.1680           0.0410       0.2441
4  0.1918           0.0411       0.2140


A few records with low NNDR are not necessarily a problem. The key question is whether the corresponding real records are themselves common , if many real individuals share similar attributes, a matching synthetic record cannot be traced back to any one of them. That is the *hiding in the crowd* effect.

## ✅ Summary

In this notebook you:

1. **Checked for exact duplicates** — the simplest form of memorisation, where a synthetic record is a verbatim copy of a real one
2. **Computed NNDR** — a distance-based metric that catches near-memorisation: synthetic records that are much closer to one real record than any other
3. **Examined a complite privacy report**

Privacy evaluation is about understanding risk and deciding what level of similarity is acceptable for your application.